In [3]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

app_path = Path.cwd().parent
sys.path.insert(0, str(app_path))

import models

db_path = Path.cwd().parents[1] / "test.db"
engine = create_async_engine(f"sqlite+aiosqlite:///{db_path}")
Session = async_sessionmaker(engine, expire_on_commit=False)

print(f"Using database: {db_path}")


Using database: /home/francisco/MAIN_2026/stock_analysis_platform_11_03_2026/Stock_Platform_v2/Portfolio-Analytics/backend/test.db


In [4]:
async def get_portfolio_transactions(db, portfolio_id: int, user_id: int):
    result = await db.execute(
        select(models.Transaction)
        .join(
            models.Portfolio,
            models.Portfolio.id == models.Transaction.portfolio_id,
        )
        .where(
            models.Portfolio.id == portfolio_id,
            models.Portfolio.user_id == user_id,
        )
        .order_by(
            models.Transaction.transaction_date,
            models.Transaction.id,
        )
    )
    return result.scalars().all()

In [5]:
portfolio_id = 1
user_id = 1

async with Session() as db:
    transactions = await get_portfolio_transactions(db, portfolio_id, user_id)

transaction_rows = [
    {
        "id": transaction.id,
        "symbol": transaction.symbol,
        "transaction_type": transaction.transaction_type.value,
        "quantity_actions": transaction.quantity_actions,
        "price": transaction.price,
        "total_value": transaction.total_value,
        "transaction_date": transaction.transaction_date,
        "portfolio_id": transaction.portfolio_id,
    }
    for transaction in transactions
]

print(f"Found {len(transaction_rows)} transactions")
pd.DataFrame(transaction_rows)


Found 4 transactions


,id,symbol,transaction_type,quantity_actions,price,total_value,transaction_date,portfolio_id
0,1,AAPL,BUY,5,10.00,50.00,2026-09-18 18:43:52.653408,1
1,2,AAPL,BUY,20,100.00,2000.00,2026-09-19 18:39:29.602198,1
2,3,META,BUY,2,0.01,0.02,2026-09-19 18:39:48.685618,1
3,4,META,SELL,1,2.00,2.00,2026-09-19 18:40:20.038119,1


In [17]:
from holding_service import HoldingService

holding_service = HoldingService(session=None)
holdings = holding_service.build_holdings_dictionary(transactions)

print(f"Found {len(holdings)} holdings")
pd.DataFrame(holdings.values())

list(holdings.values())


Found 2 holdings


[{'symbol': 'AAPL',
  'number_current_shares': 25,
  'avg_cost_per_share': 82.0,
  'cost_bases': 2050.0},
 {'symbol': 'META',
  'number_current_shares': 1,
  'avg_cost_per_share': 0.01,
  'cost_bases': 0.01}]